# 👟 ARShoe M1 Fine-Tuning on dataset/shuffled_v4

This notebook fine-tunes the trained **ARShoe M1** model (`outputs/m1_training/checkpoints/best.pth`) on the expanded **`dataset/shuffled_v4`** dataset (1,156 training images, 143 validation images, 146 test images).

### ⚙️ Fine-Tuning Strategy:
- **Pretrained Checkpoint:** Epoch 166 baseline on v3 (clean `_orig_mod.` stripping)
- **Learning Rate:** `2e-4` with Cosine Annealing decay down to `1e-6`
- **Epochs:** 75 epochs
- **Batch Size:** 16
- **Loss Weights:** Heatmap=4.0, PAF=2.0, Class=1.5 (AdaptiveWingLoss + Masked Smooth L1)
- **Output Directory:** `outputs/m1_finetune_v4/` (preserving original checkpoint!)

## Step 1: GPU Verification & Setup

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive'
print(f"Drive mounted. Available files: {os.listdir(DRIVE_ROOT)[:5]}")

## Step 3: Extract Code and Dataset into /content
Upload `shoes_vto_finetune.zip` and `shuffled_v4.zip` to your Google Drive root or directly into `/content`.

In [ ]:
import os, glob

# 1. Extract shoes_vto
vto_zip = None
for path in ['/content/drive/MyDrive/shoes_vto_finetune.zip', '/content/shoes_vto_finetune.zip']:
    if os.path.exists(path):
        vto_zip = path
        break

if vto_zip:
    print(f"📦 Extracting {vto_zip}...")
    !unzip -q -o "{vto_zip}" -d /content/
    print("✅ Code extracted successfully!")
else:
    print("ℹ️ No zip found; checking existing /content folders...")

# 2. Extract dataset/shuffled_v4
data_zip = None
for path in ['/content/drive/MyDrive/shuffled_v4.zip', '/content/shuffled_v4.zip']:
    if os.path.exists(path):
        data_zip = path
        break

!mkdir -p /content/dataset
if data_zip:
    print(f"📦 Extracting {data_zip}...")
    !unzip -q -o "{data_zip}" -d /content/dataset/
    print("✅ Dataset extracted successfully!")

# Auto-locate working directory
VTO_DIR = None
for root, dirs, files in os.walk('/content'):
    if 'evaluate_m1.py' in files and 'src' in dirs:
        VTO_DIR = root
        break

DATASET_ROOT = None
for root, dirs, files in os.walk('/content'):
    if root.endswith('shuffled_v4') and 'train' in dirs:
        DATASET_ROOT = root
        break

print(f"\n📁 Working Code Dir: {VTO_DIR}")
print(f"📁 Dataset Root Dir: {DATASET_ROOT}")

## Step 4: Install Dependencies

In [ ]:
import os
os.chdir(VTO_DIR)
!pip install -q -r requirements.txt

## Step 5: (Optional) Pre-Evaluation Baseline on shuffled_v4
Record initial metrics before fine-tuning:

In [ ]:
import os
os.chdir(VTO_DIR)
!python evaluate_m1.py \
    --checkpoint outputs/m1_training/checkpoints/best.pth \
    --dataset_root "{DATASET_ROOT}" \
    --output_dir outputs/v4_baseline_eval \
    --num_vis 5

## Step 6: 🚀 Run Fine-Tuning on dataset/shuffled_v4 (Phase 3)

In [ ]:
import os
os.chdir(VTO_DIR)
!python fine_tune_m1.py \
    --pretrained outputs/m1_training/checkpoints/best.pth \
    --dataset_root "{DATASET_ROOT}" \
    --output_dir outputs/m1_finetune_v4 \
    --epochs 75 \
    --batch_size 16 \
    --lr 0.0002 \
    --eta_min 0.000001 \
    --num_workers 4

## Step 7: Post-Fine-Tuning Evaluation & Comparative Visualizations
Evaluate the new `best.pth` checkpoint and produce 3-panel overlays:

In [ ]:
import os
os.chdir(VTO_DIR)
!python evaluate_m1.py \
    --checkpoint outputs/m1_finetune_v4/checkpoints/best.pth \
    --dataset_root "{DATASET_ROOT}" \
    --output_dir outputs/v4_finetuned_eval \
    --num_vis 10

## Step 8: Backup Fine-Tuned Checkpoints to Google Drive

In [ ]:
import os, shutil
os.chdir(VTO_DIR)
!cp outputs/m1_finetune_v4/checkpoints/best.pth /content/drive/MyDrive/best_finetuned_v4.pth
!cp -r outputs/v4_finetuned_eval /content/drive/MyDrive/v4_finetuned_eval
print("✅ Fine-tuned checkpoint and evaluation artifacts saved to Google Drive!")